# 04 — Model training and evaluation

Four models, one winner, and an operating threshold chosen on economics rather than habit.

**Target:** did an active customer cancel in the three months after the cutoff?
**Base rate:** 5.5%, so accuracy is meaningless — predicting "nobody leaves" scores 94.5%.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 60)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.color": "#EAE6DF",
                     "axes.facecolor": "white", "figure.facecolor": "white",
                     "axes.edgecolor": "#C9C4BA", "axes.axisbelow": True,
                     "legend.frameon": False})
SLATE, CLAY, AMBER = "#335C67", "#9E2A2B", "#E09F3E"


In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score
from model import get_models, evaluate, lift_at_k, business_threshold, recall_first_threshold
from preprocessing import FEATURE_COLUMNS_NUMERIC, FEATURE_COLUMNS_CATEGORICAL

df = pd.read_parquet("../data/processed/features_train.parquet")
feature_cols = FEATURE_COLUMNS_NUMERIC + FEATURE_COLUMNS_CATEGORICAL
X, y = df[feature_cols], df.churned.astype(int)

X_tmp, X_test, y_tmp, y_test, rev_tmp, rev_test = train_test_split(
    X, y, df.avg_monthly_spend.to_numpy(), test_size=0.20, stratify=y, random_state=42)
X_train, X_val, y_train, y_val, _, rev_val = train_test_split(
    X_tmp, y_tmp, rev_tmp, test_size=0.20, stratify=y_tmp, random_state=42)

print(f"train {len(X_train):,} | valid {len(X_val):,} | test {len(X_test):,}")
print(f"positive rate {y.mean():.2%}  ({int(y.sum()):,} churners)")

### Handling the imbalance

Class weights, not resampling. SMOTE on a 5% positive rate manufactures synthetic minority
points in a space where the minority is genuinely rare, which inflates validation scores
without improving the *ranking* — and ranking is what a retention desk consumes, because
they work a prioritised list, not a binary flag.

In [ ]:
pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())
print(f"scale_pos_weight = {pos_weight:.1f}")

models = get_models(pos_weight)
rows, fitted, val_probas = [], {}, {}

for name, pipe in models.items():
    if name == "Gradient Boosting":
        pipe.fit(X_train, y_train, clf__sample_weight=np.where(y_train == 1, pos_weight, 1.0))
    else:
        pipe.fit(X_train, y_train)
    p = pipe.predict_proba(X_val)[:, 1]
    fitted[name], val_probas[name] = pipe, p
    m = evaluate(y_val, p, 0.5)
    m["model"] = name
    m["lift_top10pct"] = lift_at_k(y_val, p, 0.10)
    rows.append(m)
    print(f"  {name:<20} ROC-AUC {m['roc_auc']:.3f}  PR-AUC {m['pr_auc']:.3f}")

comparison = (pd.DataFrame(rows).set_index("model")
              [["roc_auc", "pr_auc", "recall", "precision", "f1", "accuracy", "brier", "lift_top10pct"]]
              .sort_values("pr_auc", ascending=False))
comparison.round(4)

**PR-AUC, not ROC-AUC, decides the winner.** With a 5% positive class, ROC-AUC is
flattered by the enormous number of true negatives; precision-recall area reflects what the
retention desk actually experiences when they work a flagged list.

Note the logistic regression is not embarrassed here. It is within a couple of points of the
boosted models on ROC-AUC, which is a useful reminder that most of the work was done in
feature engineering, not in model choice.

In [ ]:
best_name = comparison.index[0]
best_model = fitted[best_name]
print(f"Selected: {best_name}")

cv = cross_val_score(get_models(pos_weight)[best_name], X_tmp, y_tmp,
                     cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="roc_auc")
print(f"5-fold CV ROC-AUC: {cv.mean():.4f} +/- {cv.std():.4f}  ->  {np.round(cv, 4)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for (name, p), c in zip(val_probas.items(), [SLATE, CLAY, AMBER, "#6B9080"]):
    fpr, tpr, _ = roc_curve(y_val, p)
    axes[0].plot(fpr, tpr, color=c, lw=2, label=f"{name} ({roc_auc_score(y_val, p):.3f})")
    prec, rec, _ = precision_recall_curve(y_val, p)
    axes[1].plot(rec, prec, color=c, lw=2, label=f"{name} ({average_precision_score(y_val, p):.3f})")
axes[0].plot([0, 1], [0, 1], ls="--", color="#C9C4BA")
axes[0].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC curves")
axes[0].legend(fontsize=8, loc="lower right")
axes[1].axhline(y_val.mean(), ls="--", color="#C9C4BA")
axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision-recall curves")
axes[1].legend(fontsize=8)
plt.tight_layout()

## The threshold is a business decision

0.5 is an arbitrary default that assumes a false positive and a false negative cost the same.
They do not:

- **False negative** — a customer leaves. We lose the margin on their remaining lifetime.
- **False positive** — we make a retention offer to someone who was staying anyway. We lose
  the cost of the offer.

Those differ by more than an order of magnitude, so the threshold is chosen by maximising
expected retained margin on the validation set:

```
value of a catch = monthly revenue x gross margin x expected lifetime x save rate
cost of a flag   = monthly revenue x offer discount x months honoured
```

In [ ]:
p_val = val_probas[best_name]
thr_business, ev_curve = business_threshold(y_val, p_val, rev_val)
thr_recall = recall_first_threshold(y_val, p_val, target_recall=0.85)
best_row = ev_curve.loc[ev_curve.expected_value.idxmax()]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ev_curve.threshold, ev_curve.expected_value / 1e3, color=SLATE, lw=2.4)
ax.axvline(thr_business, color=CLAY, ls="--")
ax.set(xlabel="decision threshold", ylabel="expected retained margin ($K)",
       title="Where an extra retention call stops paying for itself")
plt.tight_layout()

print(f"Business-optimal threshold : {thr_business:.2f}  "
      f"({int(best_row.flagged):,} flagged, recall {best_row.recall:.0%}, precision {best_row.precision:.0%})")
print(f"Recall-first threshold     : {thr_recall:.3f}")

## Final evaluation — the test set, touched once

Everything above was decided on validation data. The test set is scored a single time, at
three operating points.

In [ ]:
p_test = best_model.predict_proba(X_test)[:, 1]
results = pd.DataFrame({
    "default (0.5)":     evaluate(y_test, p_test, 0.5),
    "business-optimal":  evaluate(y_test, p_test, thr_business),
    "recall-first":      evaluate(y_test, p_test, thr_recall),
}).T[["threshold", "recall", "precision", "f1", "roc_auc", "pr_auc", "accuracy", "tp", "fp", "fn", "tn"]]
results.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (label, thr) in zip(axes, [("Default", 0.5), ("Business-optimal", thr_business),
                                   ("Recall-first", thr_recall)]):
    pred = (p_test >= thr).astype(int)
    cm = np.array([[((pred == 0) & (y_test == 0)).sum(), ((pred == 1) & (y_test == 0)).sum()],
                   [((pred == 0) & (y_test == 1)).sum(), ((pred == 1) & (y_test == 1)).sum()]])
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", fontsize=13,
                    color="white" if cm[i, j] > cm.max() / 2 else "#22333B")
    rec = cm[1, 1] / cm[1].sum()
    prec = cm[1, 1] / max(cm[:, 1].sum(), 1)
    ax.set_title(f"{label} (thr {thr:.2f})\nrecall {rec:.0%} | precision {prec:.0%}", fontsize=10)
    ax.set_xticks([0, 1], ["pred. stay", "pred. churn"])
    ax.set_yticks([0, 1], ["stayed", "churned"])
    ax.grid(False)
plt.tight_layout()

## Does the ranking work?

The desk cannot call everyone. What matters operationally is how much of the churn is
concentrated at the top of the list.

In [ ]:
order = np.argsort(-p_test)
y_sorted = y_test.to_numpy()[order]
capture = np.cumsum(y_sorted) / y_sorted.sum()
pct = np.arange(1, len(y_sorted) + 1) / len(y_sorted)

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(pct * 100, capture * 100, color=SLATE, lw=2.6, label="Model")
ax.plot([0, 100], [0, 100], ls="--", color="#C9C4BA", label="Random targeting")
for k in (0.05, 0.10, 0.20):
    ax.scatter([k * 100], [capture[int(k * len(y_sorted))] * 100], color=CLAY, zorder=5)
    ax.annotate(f"top {k:.0%}: {capture[int(k*len(y_sorted))]:.0%} of churners",
                (k * 100, capture[int(k * len(y_sorted))] * 100),
                textcoords="offset points", xytext=(10, -12), fontsize=9)
ax.set(xlabel="% of customers contacted, highest risk first",
       ylabel="% of churners captured", title="Cumulative gains")
ax.legend()
plt.tight_layout()

Contacting the riskiest 10% of the book reaches roughly three quarters of everyone who is
about to leave. That is the number to quote to a retention manager — it converts directly
into a staffing plan.

## Calibration

The probability has to *mean* something, because CLV and revenue-at-risk multiply by it.

In [ ]:
bins = np.linspace(0, 1, 11)
idx = np.digitize(p_test, bins) - 1
obs = [y_test.to_numpy()[idx == b].mean() if (idx == b).sum() > 20 else np.nan for b in range(10)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot([0, 1], [0, 1], ls="--", color="#C9C4BA")
axes[0].plot(bins[:-1] + 0.05, obs, marker="o", color=CLAY, lw=2)
axes[0].set(xlabel="predicted probability", ylabel="observed churn rate", title="Calibration")
axes[1].hist(p_test[y_test == 0], bins=40, color=SLATE, alpha=.75, label="stayed")
axes[1].hist(p_test[y_test == 1], bins=40, color=CLAY, alpha=.75, label="churned")
axes[1].set_yscale("log")
axes[1].set(xlabel="predicted probability", title="Score separation")
axes[1].legend()
plt.tight_layout()

**Next:** `05_SHAP_analysis.ipynb` — turning the score into a reason and an action.